In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv("data/candidate_job_market_dataset_processed.csv")

In [3]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [4]:
import sys
print(sys.executable)
print("\n".join(sys.path))

c:\ProgramData\anaconda3\python.exe
c:\ProgramData\anaconda3\python314.zip
c:\ProgramData\anaconda3\DLLs
c:\ProgramData\anaconda3\Lib
c:\ProgramData\anaconda3

C:\Users\vartika singh\AppData\Roaming\Python\Python314\site-packages
c:\ProgramData\anaconda3\Lib\site-packages
c:\ProgramData\anaconda3\Lib\site-packages\win32
c:\ProgramData\anaconda3\Lib\site-packages\win32\lib
c:\ProgramData\anaconda3\Lib\site-packages\pythonwin


In [6]:
from sklearn.cluster import KMeans
print("KMeans working")

ImportError: DLL load failed while importing _argkmin: An Application Control policy has blocked this file.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
tfidf=TfidfVectorizer()
tfidf.fit(pd.concat([df["candidate_skills"],df["required_skills"]]).fillna("").drop_duplicates())

ImportError: DLL load failed while importing _argkmin: An Application Control policy has blocked this file.

In [ ]:
applicant=df.drop_duplicates('candidate_id').reset_index(drop=True)
B=tfidf.transform(applicant['candidate_skills'])

In [ ]:
applicant["job_title"].unique()

array(['Machine Learning Intern', 'Data Analyst Intern',
       'Python Developer Intern', 'AI Intern', 'Backend Developer Intern',
       'Frontend Developer Intern', 'Data Science Intern',
       'Software Engineer Intern'], dtype=object)

In [ ]:
from sklearn.metrics import silhouette_score 
score={}
for k in range(2,15):
    km_k=KMeans(n_clusters=k,random_state=42,n_init=12)
    label=km_k.fit_predict(B)
    score[k]=silhouette_score(B,label,sample_size=1000,random_state=42)

best_k=max(score,key=score.get)


In [ ]:
km=KMeans(n_clusters=best_k,random_state=42,n_init=12)
applicant['cluster']=km.fit_predict(B)

In [ ]:
best_k

12

In [ ]:
df=df.merge(applicant[["candidate_id","cluster"]],on="candidate_id",how="left")

In [ ]:
cluster_role={}
for c in range(best_k):
    top=df[(df.cluster==c)& (df.fit_label==1)][['job_title','required_skills']].drop_duplicates("job_title")
    cluster_role[c]=top["job_title"].tolist()

In [ ]:
jobs = df.drop_duplicates('job_title')[['job_title','required_skills']].reset_index(drop=True)


In [ ]:
job_vecs=tfidf.transform(jobs["required_skills"].fillna(""))

In [ ]:
joblib.dump({'tfidf':tfidf,'km':km,'cluster_roles' :cluster_role,'jobs':jobs,"job_vecs":job_vecs},'recommender.pkl')

['recommender.pkl']